# Trace and validate the pipeline

Linden Facilities needs 24 hourly estimates. Follow the five stage contracts before making a bounded validation change.

Run all cells top to bottom. **Colab:** connect a fresh CPU / Python 3.12 runtime; the first cell clones the pinned GitHub source and installs its locked runtime dependencies. **Local:** run `uv sync --locked` from `module-a` and select that `.venv` interpreter. See [Colab and environment guidance](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/colab.md). Every run creates a new temporary tracking directory and prints its path; download evidence before ending a hosted runtime. No API key is needed. These are complete worked references; independent practices have separate starting states. IDs and timing vary.

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'eec754e0b1379fe8bb01e23543b92e036b4c6b62', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    for name, version in expected.items():
        if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version:
            raise RuntimeError(f'{name} is already loaded at another version. Delete/reconnect the runtime, then run setup first.')
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: eec754e0b1379fe8bb01e23543b92e036b4c6b62 state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-ngccq1ad


## Inspect the stage boundaries
`ingest` reads the fixed snapshot. Features exclude target and partition. Expect 168 train, 48 validation and 24 final rows.

In [2]:
frame=ingest()
print(frame.groupby('partition').size())
display(frame.head())

partition
final          24
train         168
validation     48
dtype: int64


,row_id,temperature_c,hour,occupancy,demand_kwh,partition
0,D000,21.714,0,89,76.066729,validation
1,D001,-1.100,1,0,45.455758,validation
2,D002,9.338,2,12,44.943443,final
3,D003,16.650,3,67,69.456248,train
4,D004,27.623,4,34,45.034036,train


## Trace preparation, fitting and evaluation
The scaler belongs to the fitted model, so it sees training rows only. RMSE measures larger errors more strongly; MAE is diagnostic. Expect baseline validation RMSE about 1.443003925 kWh.

In [3]:
from campus_mlops.pipeline import TARGET
config=Config()
features=preprocess(frame,config)
train_rows=frame.partition=='train'
valid_rows=frame.partition=='validation'
model=train(features.loc[train_rows],frame.loc[train_rows,TARGET],config)
metrics=evaluate(model,features.loc[valid_rows],frame.loc[valid_rows,TARGET])
print(features.columns.tolist(), metrics)
assert abs(metrics['rmse_kwh']-1.4430039250255964)<1e-6

['temperature_c', 'hour', 'occupancy'] {'rmse_kwh': 1.4430039250255964, 'mae_kwh': 1.2234272153578762}


## Read the extension, then test both branches
This is the worked reference. Explain every line before using it. It rejects invalid values and copies valid inputs. Enabling it must preserve valid outputs.

In [4]:
import inspect
print(inspect.getsource(validate_ranges))
import pandas as pd
pd.testing.assert_frame_equal(validate_ranges(frame),frame)
broken=frame.copy(); broken.loc[0,'hour']=24
try:
    validate_ranges(broken)
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('Invalid hour was accepted')
assert run_pipeline(Config(range_guard=True))[1]==metrics

def validate_ranges(frame):
    """Bounded extension: reject invalid readings; never clip/drop/reorder valid rows."""
    validate_features(frame)
    for column, low, high in [("temperature_c", -30, 50), ("hour", 0, 23), ("occupancy", 0, 100)]:
        if not frame[column].between(low, high).all():
            raise ValueError(f"{column} outside [{low}, {high}]")
    if not (frame.hour % 1 == 0).all():
        raise ValueError("hour must be an integer")
    return frame.copy(deep=True)

Expected rejection: hour outside [0, 23]


## Complete the inference stage
The prepared bootstrap uses the same pipeline, logs it, registers and activates an approved model. Expect 24 finite output rows tied to the resolved registry version. Timing is observed, not a fixed benchmark.

In [5]:
ws=Workspace(state/'tracking')
record=ws.run(Config(range_guard=True),name='guarded-baseline')
version=register(ws,record['run_id'])
approve(ws,version,'Valid/rejected input checks and RMSE <= 5')
activate(ws,version,'Baseline for batch demonstration')
predictions,evidence=batch_predict(ws)
display(predictions.head()); print(evidence)
assert len(predictions)==24 and evidence['resolved_version']==version
assert evidence['complete_within_60s']

2026/10/19 14:42:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 14:42:11 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:11 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:11 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 14:42:11 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026-10-19 14:42:11 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:11 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Successfully registered model 'dev.linden-demand'.
Created version '1' of model 'dev.linden-demand'.


,row_id,prediction_kwh
0,B00,47.639043
1,B01,48.413678
2,B02,49.188314
3,B03,49.962949
4,B04,50.737584


{'resolved_version': '1', 'run_id': '7f85178fb090455f86917f3ca49f562c', 'model': 'dev.linden-demand', 'rows': 24, 'elapsed_seconds': 0.008177375013474375, 'rows_per_second': 2934.927156997654, 'complete_within_60s': True, 'input_sha256': '5b74deb237ea9a085666ab0b3f865db8c7c996b935b38b24e829567d89142ee6'}
